In [1]:
import pandas as pd
import mario
mario.set_log_verbosity('critical')
import country_converter as coco

ghgs = pd.DataFrame()

shared_folder = '/Users/lorenzorinaldi/Library/CloudStorage/OneDrive-SharedLibraries-PolitecnicodiMilano/DENG-SESAM - Documenti/c-Research/a-Datasets/_Input Output Databases'
properties = {
    'exiobase_monetary': {
        'path': f'{shared_folder}/EXIOBASE/{{version}}/{{table}}_{{year}}_ixi.zip',
        'name': 'EXIOBASE', 'table': 'IOT', 'versions': ['3.10.2'], 'years': [2010,2015,2018,2021],
    },
    'eora': {
        'path': f'{shared_folder}/EORA/EORA1/Eora26/Eora26_{{year}}_bp',
        'name': 'EORA26','table': 'IOT','versions': ['199.82'],'years': [2010,2015],
    },
    'emerging': {
        'path': f'{shared_folder}/EMERGING/2.2/EMERGING_V2_{{year}}_m.mat',
        'name': 'EMERGING','table': 'IOT','versions': ['2.2'],'years': [2015, 2018, 2021, 2023],
    },
    'adb': {
        'path': f'{shared_folder}/ADB/{{version}}',
        'path_satellites': f'{shared_folder}/ADB/Air emissions/{{year}} EE-MRIOT (Air Emissions).xlsx',
        'name': 'ADB','table': 'IOT','versions': ['62 economies'],'years': [2018,2021,2023],
    },
    'gloria': {
        'path': f'{shared_folder}/GLORIA',
        'name': 'GLORIA','table': 'SUT','versions': ['0.60'],'years': [2015, 2018, 2021, 2023],
    },
}

def ghg_rearrangement(db, name, version, table, year):
        df_pba = db.E.loc["GHG", :].T.groupby(level=0).sum() + db.EY.loc["GHG", :].T.groupby(level=0).sum()
        df_cba = db.F.loc["GHG", :].T.groupby(level=0).sum() + db.EY.loc["GHG", :].T.groupby(level=0).sum()
        df_pba = df_pba.to_frame()
        df_cba = df_cba.to_frame()
        unit = db.units['Satellite account'].loc['GHG','unit']
        df_pba.columns = pd.MultiIndex.from_tuples([(name, version, table, year, "PBA", unit)], names=['database', 'version', 'table', 'year', 'accounting', 'unit'])
        df_cba.columns = pd.MultiIndex.from_tuples([(name, version, table, year, "CBA", unit)], names=['database', 'version', 'table', 'year', 'accounting', 'unit'])
        df = pd.concat([df_pba, df_cba], axis=1)
        return df

def reconcile_to_iso3(df):
    cc = coco.CountryConverter()
    iso3_index = cc.convert(df.index.tolist(), to='ISO3', not_found=None)
    df.index = iso3_index
    df.index.name = 'Region'
    return df

## EXIOBASE

In [ ]:
p = properties['exiobase_monetary']
name, table, path = p['name'], p['table'], p['path']

for version in p['versions']:
    for year in p['years']:
        print(f'=== {name} {version} {table} ixi {year} ===')

        db = mario.parse_exiobase(
            path=path.format(version=version, year=year, table=table),
            table=table,
            unit='Monetary',
        )

        db.calc_ghg(profile='exiobase_monetary')
        ghgs_df = ghg_rearrangement(db, name, version, table, year)
        ghgs_df = reconcile_to_iso3(ghgs_df)
        ghgs = pd.concat([ghgs, ghgs_df], axis=1)

## EORA26

In [ ]:
p = properties['eora']
name, table, path = p['name'], p['table'], p['path']

for version in p['versions']:
    for year in p['years']:
        print(f'=== {name} {version} {table} ixi {year} ===')

        db = mario.parse_eora(
            path=path.format(year=year),
            multi_region=True,
        )

        db.calc_ghg(profile='eora')
        ghgs_df = ghg_rearrangement(db, name, version, table, year)
        ghgs_df = reconcile_to_iso3(ghgs_df)
        ghgs = pd.concat([ghgs, ghgs_df], axis=1)

## EMERGING

In [ ]:
p = properties['emerging']
name, table, path = p['name'], p['table'], p['path']

for version in p['versions']:
    for year in [2018]:#p['years']:
        print(f'=== {name} {version} {table} ixi {year} ===')

        db = mario.parse_emerging(
            path=path.format(year=year),
            year=year,
        )

        db.calc_ghg(profile='emerging')
        ghgs_df = ghg_rearrangement(db, name, version, table, year)
        ghgs_df = reconcile_to_iso3(ghgs_df)
        ghgs = pd.concat([ghgs, ghgs_df], axis=1)

## ADB

In [ ]:
p = properties['adb']
name, table, path = p['name'], p['table'], p['path']

for version in p['versions']:
    for year in [2018]:#p['years']:
        print(f'=== {name} {version} {table} ixi {year} ===')

        db = mario.parse_adb(
            path=path.format(version=version),
            year=year,
            add_extensions=p['path_satellites'].format(year=year),
        )

        db.calc_ghg(profile='adb')
        ghgs_df = ghg_rearrangement(db, name, version, table, year)
        ghgs_df = reconcile_to_iso3(ghgs_df)
        ghgs = pd.concat([ghgs, ghgs_df], axis=1)

=== ADB 62 economies IOT ixi 2018 ===


In [ ]:
sat = "GHG | Total by substance"
print(db.E.loc[sat, :].T.groupby(level=0).sum().loc["ITA"])
print(db.EY.loc[sat, :].T.groupby(level=0).sum().loc["ITA"])


## GLORIA

In [ ]:
p = properties['gloria']
name, table, path = p['name'], p['table'], p['path']

for version in p['versions']:
    for year in p['years']:
        print(f'=== {name} {version} {table} ixi {year} ===')

        db = mario.parse_gloria(
            path=path,
            year=year,
        )

        db.calc_ghg(profile='gloria')
        ghgs_df = ghg_rearrangement(db, name, version, table, year)
        ghgs_df = reconcile_to_iso3(ghgs_df)
        ghgs = pd.concat([ghgs, ghgs_df], axis=1)